# Research 3 - narrow positive bridge replay

Run setup, mount, restore, actual-data audit, matched fits, and export in order.
One A10080GB; ONE resident model worker, reference then bridge, both from
preserved H2 with fresh optimizers. Only nine positive replay rows change;
B80 plus23 replay rows, including all factual/preference/ordinary rows, stay fixed.
The original checkpoint and known-development panels remain unchanged. Final12
singleton handoff cases are supplied-history DEVELOPMENT DIAGNOSTICS.
No new families or confirmation, control fits, triggers, steering or monitoring.
Models use fictional memory tools only. Drive is unmounted during model work.

Cumulative authorization200units;94.31 previously accounted; refresh actual
credit/rate. Stagecap24 includes2 export/release reserve. Main cap10,800seconds
is reduced by elapsed billed setup time. Export verifies every private file and
then releases the runtime. Existing weights, failures and Research2 remain intact.


In [ ]:
import gzip,hashlib,json,subprocess,sys,time,uuid
from pathlib import Path,PurePosixPath
from urllib.request import urlopen
SESSION_STARTED=time.monotonic()
BALANCE_AT_LAUNCH=107.82
RATE_AT_LAUNCH=6.77
PRIOR_SPEND=94.31
AUTHORIZED_TOTAL_UNITS=200
STAGE_CAP_UNITS=24
RESERVE_UNITS=2
assert STAGE_CAP_UNITS<=min(BALANCE_AT_LAUNCH,AUTHORIZED_TOTAL_UNITS-PRIOR_SPEND)
gpu=subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                   capture_output=True,text=True,check=True).stdout
assert 'A100' in gpu and int(gpu.split(',')[1].split()[0])>=75000,gpu
ROOT=Path('/content/sp_lense_work')/('qwen38_narrow_bridge_replay_'+
    time.strftime('%Y%m%dT%H%M%SZ',time.gmtime())+'_'+uuid.uuid4().hex[:8])
ROOT.mkdir(parents=True,exist_ok=False)
SOURCE_REVISION='ac1a3af71e2f9cf53da68beddb3f55a2b8e87bd4'
SOURCE_URL='https://raw.githubusercontent.com/Farhad-Davaripour/sp_lense/ac1a3af71e2f9cf53da68beddb3f55a2b8e87bd4/study/03_open_agent_self_preservation/qwen38_colab/narrow_bridge_replay_v1/source_bundle.json.gz'
SOURCE_BUNDLE_SHA256='bbffc9668ce7c251dcd7de5aad2255d1c02aed2089a9fe66f8689c1614191498'
with urlopen(SOURCE_URL,timeout=120) as response:packed=response.read()
assert len(packed)==128136 and hashlib.sha256(packed).hexdigest()==SOURCE_BUNDLE_SHA256
payload=json.loads(gzip.decompress(packed));assert payload['schema_version']==1
FILES=payload['files']
for name,content in FILES.items():
    relative=PurePosixPath(name)
    assert not relative.is_absolute() and '..' not in relative.parts
    path=ROOT/'source'/name;path.parent.mkdir(parents=True,exist_ok=True)
    path.write_bytes(content.encode('utf-8'))
for name,digest in json.loads(FILES['SOURCE_FREEZE.json'])['sha256'].items():
    assert hashlib.sha256((ROOT/'source'/name).read_bytes()).hexdigest()==digest,name
(ROOT/'source_bundle.json.gz').write_bytes(packed)
(ROOT/'SOURCE_BUNDLE_DOWNLOAD.json').write_text(json.dumps({'revision':SOURCE_REVISION,
    'url':SOURCE_URL,'sha256':SOURCE_BUNDLE_SHA256,'bytes':len(packed)},indent=2))
subprocess.run([sys.executable,'-m','pip','install','--quiet','--no-input','-r',
                str(ROOT/'source/requirements.txt')],check=True,timeout=900)
sys.path.insert(0,str(ROOT/'source'))
print('NARROW_REPLAY_SOURCE_READY',ROOT,gpu,'authorized remaining',AUTHORIZED_TOTAL_UNITS-PRIOR_SPEND,flush=True)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('DRIVE_MOUNT_CONNECTED',flush=True)


In [ ]:
from restore_inputs import restore_inputs,download_base
MODEL_INPUTS=restore_inputs(ROOT)
drive.flush_and_unmount()
print('DRIVE_UNMOUNTED_BEFORE_MODEL_WORK',flush=True)
BASE_MANIFEST=download_base(ROOT)


In [ ]:
from prepare_data import prepare
WORKER_ROOT,DATA_AUDIT=prepare(ROOT,MODEL_INPUTS)
print('ACTUAL_DATA_AUDIT',json.dumps({key:value for key,value in DATA_AUDIT.items()
    if key not in ('paired_lengths','content_audit')},indent=2))
print('EXACT_CHANGED_ROWS',json.dumps(DATA_AUDIT['content_audit']['changes'],indent=2))


In [ ]:
exec(compile((ROOT/'source/launch.py').read_text(),'trusted_narrow_replay.py','exec'))


In [ ]:
import re
for worker in ROOT.rglob('worker.py'):
    assert not subprocess.run(['pgrep','-f',re.escape(str(worker))],capture_output=True,text=True).stdout.strip()
from collect import collect
REVIEW_EVIDENCE=collect(ROOT)
FAST_ROOT=ROOT
EXPORT_ONLY_ROOTS=[ROOT]
RELEASE_RUNTIME_AFTER_EXPORT=True
exec(compile((ROOT/'source/export_private_runs.py').read_text(),'trusted_private_export.py','exec'))
